# 05bis - Exploration Dataset Kaggle : Premier League 1992-2024

**Objectif** : Explorer le dataset Kaggle "All Premier League Team and Players 1992-2024" pour identifier les variables utiles susceptibles d'enrichir notre modélisation (xG, valeurs marchandes, stats joueurs, etc.).

**Dataset source** : `samoilovmikhail/all-premier-league-team-and-players-1992-2024`

**Plan d'exploration** :
1. Téléchargement et inventaire des fichiers disponibles
2. Inspection structure et période couverte
3. Identification des variables pertinentes (xG, valeurs, lineups)
4. Évaluation compatibilité avec notre dataset existant (pl_features_v1.csv)
5. Décision : intégration directe OU scraping complémentaire

---

In [1]:
# Cellule 1 : Téléchargement et inventaire du dataset Kaggle

import kagglehub
import os
from pathlib import Path

# Téléchargement du dataset
print("📥 Téléchargement du dataset Kaggle...")
path = kagglehub.dataset_download("samoilovmikhail/all-premier-league-team-and-players-1992-2024")
print(f"✅ Dataset téléchargé dans : {path}\n")

# Inventaire des fichiers disponibles
print("=" * 80)
print("📋 INVENTAIRE DES FICHIERS")
print("=" * 80)

csv_files = []
json_files = []
other_files = []

for root, dirs, files in os.walk(path):
    for file in files:
        file_path = Path(root) / file
        file_size = file_path.stat().st_size / (1024 * 1024)  # Taille en MB
        
        if file.endswith('.csv'):
            csv_files.append((file, file_size))
        elif file.endswith('.json'):
            json_files.append((file, file_size))
        else:
            other_files.append((file, file_size))

# Affichage des CSV
print(f"\n📊 FICHIERS CSV ({len(csv_files)}) :")
print("-" * 80)
for name, size in sorted(csv_files):
    print(f"  • {name:<60} {size:>8.2f} MB")

# Affichage des JSON
print(f"\n📦 FICHIERS JSON ({len(json_files)}) :")
print("-" * 80)
for name, size in sorted(json_files):
    print(f"  • {name:<60} {size:>8.2f} MB")

# Affichage des autres fichiers
if other_files:
    print(f"\n📄 AUTRES FICHIERS ({len(other_files)}) :")
    print("-" * 80)
    for name, size in sorted(other_files):
        print(f"  • {name:<60} {size:>8.2f} MB")

# Calcul de la taille totale
total_size = sum([s for _, s in csv_files + json_files + other_files])
print("\n" + "=" * 80)
print(f"💾 TAILLE TOTALE : {total_size:.2f} MB")
print("=" * 80)

# Sauvegarde du chemin pour les cellules suivantes
dataset_path = path
print(f"\n✅ Variable 'dataset_path' créée : {dataset_path}")


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Téléchargement du dataset Kaggle...


100%|██████████| 2.95M/2.95M [00:07<00:00, 415kB/s]

Extracting files...


✅ Dataset téléchargé dans : /root/.cache/kagglehub/datasets/samoilovmikhail/all-premier-league-team-and-players-1992-2024/versions/5

📋 INVENTAIRE DES FICHIERS

📊 FICHIERS CSV (707) :
--------------------------------------------------------------------------------
  • AFC_Bournemouth_989_2015.csv                                     0.01 MB
  • AFC_Bournemouth_989_2016.csv                                     0.01 MB
  • AFC_Bournemouth_989_2017.csv                                     0.00 MB
  • AFC_Bournemouth_989_2018.csv                                     0.00 MB
  • AFC_Bournemouth_989_2019.csv                                     0.01 MB
  • AFC_Bournemouth_989_2022.csv                                     0.01 MB
  • AFC_Bournemouth_989_2023.csv                                     0.01 MB
  • AFC_Bournemouth_989_2024.csv                                     0.00 MB
  • AFC_Bournemouth_989_2025.csv                                     0.00 MB
  • AFC_Bournemouth_989_2026.csv          

In [3]:
# Cellule 2bis : Localisation et inspection des fichiers

import pandas as pd
from pathlib import Path
import os
import json

# Explorer la structure des répertoires
print("=" * 80)
print("EXPLORATION DE LA STRUCTURE DU DATASET")
print("=" * 80)
print(f"Chemin racine : {dataset_path}\n")

# Lister tous les sous-répertoires et fichiers à la racine
for item in sorted(Path(dataset_path).iterdir()):
    if item.is_dir():
        file_count = len(list(item.iterdir()))
        print(f"[DIR]  {item.name}/ ({file_count} fichiers)")
    else:
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f"[FILE] {item.name} ({size_mb:.2f} MB)")

# Trouver le premier fichier CSV et JSON (peu importe le répertoire)
csv_files = list(Path(dataset_path).rglob("*.csv"))
json_files = list(Path(dataset_path).rglob("*.json"))

print(f"\n" + "=" * 80)
print(f"FICHIERS TROUVES")
print("=" * 80)
print(f"Total CSV  : {len(csv_files)}")
print(f"Total JSON : {len(json_files)}")

# Sélectionner un exemple CSV (Arsenal 2023 si disponible, sinon le premier trouvé)
sample_csv = None
for f in csv_files:
    if "Arsenal_FC_11_2023" in f.name:
        sample_csv = f
        break
if sample_csv is None and len(csv_files) > 0:
    sample_csv = csv_files[0]

# Charger et inspecter le CSV
if sample_csv:
    print(f"\n" + "=" * 80)
    print(f"INSPECTION CSV : {sample_csv.name}")
    print("=" * 80)
    df_sample = pd.read_csv(sample_csv)
    print(f"Dimensions : {df_sample.shape[0]} lignes x {df_sample.shape[1]} colonnes")
    print(f"\nColonnes disponibles :")
    for i, col in enumerate(df_sample.columns, 1):
        print(f"  {i:2d}. {col}")
    print(f"\nApercu des 3 premieres lignes :")
    print(df_sample.head(3))
    print(f"\nTypes de donnees :")
    print(df_sample.dtypes)

# Charger et inspecter le JSON correspondant
sample_json = sample_csv.with_suffix('.json') if sample_csv else None
if sample_json and sample_json.exists():
    print(f"\n" + "=" * 80)
    print(f"INSPECTION JSON : {sample_json.name}")
    print("=" * 80)
    with open(sample_json, 'r', encoding='utf-8') as f:
        data_json = json.load(f)
    
    if isinstance(data_json, list):
        print(f"Type : Liste de {len(data_json)} elements")
        if len(data_json) > 0:
            print(f"\nCles du premier element :")
            for i, key in enumerate(data_json[0].keys(), 1):
                print(f"  {i:2d}. {key}")
            print(f"\nPremier element complet :")
            print(json.dumps(data_json[0], indent=2, ensure_ascii=False))
    elif isinstance(data_json, dict):
        print(f"Type : Dictionnaire avec {len(data_json)} cles")
        print(f"Cles racine : {list(data_json.keys())}")

print("\n" + "=" * 80)
print("Inspection terminee")
print("=" * 80)


EXPLORATION DE LA STRUCTURE DU DATASET
Chemin racine : /root/.cache/kagglehub/datasets/samoilovmikhail/all-premier-league-team-and-players-1992-2024/versions/5

[DIR]  DATA_CSV/ (35 fichiers)
[DIR]  DATA_JSON/ (35 fichiers)
[FILE] clubs.csv (0.00 MB)

FICHIERS TROUVES
Total CSV  : 707
Total JSON : 706

INSPECTION CSV : Arsenal_FC_11_2023.csv
Dimensions : 40 lignes x 17 colonnes

Colonnes disponibles :
   1. position
   2. foot
   3. status
   4. joinedOn
   5. name
   6. height
   7. id
   8. nationality
   9. marketValue
  10. signedFrom
  11. age
  12. dateOfBirth
  13. currentClub
  14. jerseyNumber
  15. ageAtSeasonStart
  16. isLoan
  17. loanedFrom

Apercu des 3 premieres lignes :
     position   foot status    joinedOn            name  height      id  \
0  Goalkeeper  right    NaN  2024-04-07      David Raya     186  262749   
1  Goalkeeper  right    NaN  2021-08-20  Aaron Ramsdale     190  427568   
2  Goalkeeper  right    NaN  2022-01-07     Matt Turner     190  425306   

   

In [4]:
# Cellule 3 : Inspection des repertoires DATA_CSV et DATA_JSON

import pandas as pd
from pathlib import Path
import json

data_csv_dir = Path(dataset_path) / "DATA_CSV"
data_json_dir = Path(dataset_path) / "DATA_JSON"

print("=" * 80)
print("CONTENU DU REPERTOIRE DATA_CSV/")
print("=" * 80)
csv_files_in_dir = sorted(list(data_csv_dir.glob("*.csv")))
print(f"Nombre de fichiers : {len(csv_files_in_dir)}\n")
for f in csv_files_in_dir[:10]:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<50} {size_kb:>8.2f} KB")
if len(csv_files_in_dir) > 10:
    print(f"  ... ({len(csv_files_in_dir) - 10} autres fichiers)")

# Inspecter un fichier exemple
if len(csv_files_in_dir) > 0:
    sample_file = csv_files_in_dir[0]
    print(f"\n" + "=" * 80)
    print(f"INSPECTION : {sample_file.name}")
    print("=" * 80)
    df_sample = pd.read_csv(sample_file)
    print(f"Dimensions : {df_sample.shape[0]} lignes x {df_sample.shape[1]} colonnes")
    print(f"\nColonnes disponibles :")
    for i, col in enumerate(df_sample.columns, 1):
        print(f"  {i:2d}. {col}")
    print(f"\nApercu des 5 premieres lignes :")
    print(df_sample.head())
    print(f"\nApercu des 5 dernieres lignes :")
    print(df_sample.tail())

print("\n" + "=" * 80)
print("CONTENU DU REPERTOIRE DATA_JSON/")
print("=" * 80)
json_files_in_dir = sorted(list(data_json_dir.glob("*.json")))
print(f"Nombre de fichiers : {len(json_files_in_dir)}\n")
for f in json_files_in_dir[:10]:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<50} {size_kb:>8.2f} KB")
if len(json_files_in_dir) > 10:
    print(f"  ... ({len(json_files_in_dir) - 10} autres fichiers)")

# Inspecter un fichier JSON correspondant
if len(json_files_in_dir) > 0:
    sample_json_file = json_files_in_dir[0]
    print(f"\n" + "=" * 80)
    print(f"INSPECTION JSON : {sample_json_file.name}")
    print("=" * 80)
    with open(sample_json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    if isinstance(data, list):
        print(f"Type : Liste de {len(data)} elements")
        if len(data) > 0:
            print(f"\nCles du premier element :")
            for key in data[0].keys():
                print(f"  - {key}")
            print(f"\nPremier element :")
            print(json.dumps(data[0], indent=2, ensure_ascii=False)[:1500])
    elif isinstance(data, dict):
        print(f"Type : Dictionnaire")
        print(f"Cles racine : {list(data.keys())}")

print("\n" + "=" * 80)
print("Inspection terminee")
print("=" * 80)


CONTENU DU REPERTOIRE DATA_CSV/
Nombre de fichiers : 0


CONTENU DU REPERTOIRE DATA_JSON/
Nombre de fichiers : 0


Inspection terminee


In [7]:
# Cellule 5 : Bilan et decision strategique

import pandas as pd
from pathlib import Path

print("=" * 80)
print("BILAN : DATASET KAGGLE vs BESOINS DU PROJET")
print("=" * 80)

print("\nCONTENU DU DATASET KAGGLE :")
print("-" * 80)
print("Structure :")
print("  - 707 fichiers CSV (equipe + saison)")
print("  - 706 fichiers JSON (identiques aux CSV)")
print("  - 1 fichier clubs.csv (mapping equipes)")
print("  - Periode couverte : 1992-2026")
print("\nDonnees disponibles par fichier :")
print("  - Liste des joueurs de l'equipe pour la saison")
print("  - Informations joueur : nom, age, poste, nationalite, taille, pied")
print("  - Valeur marchande (marketValue) en EUR")
print("  - Informations transfert : date d'arrivee, club precedent, pret")
print("\nDonnees ABSENTES :")
print("  - Stats de performance (buts, assists, minutes jouees)")
print("  - xG (Expected Goals)")
print("  - Donnees de matchs")
print("  - Stats par match ou aggregees sur la saison")

print("\n" + "=" * 80)
print("BESOINS DU PROJET (rappel)")
print("=" * 80)
print("\nVariables cibles identifiees dans PROJECT_CONTEXT.md :")
print("  1. Valeur marchande effectif (Home/Away) + delta")
print("  2. xG match")
print("  3. xG cumule saison par equipe")
print("  4. Lineups (11 titulaires + bancs)")
print("  5. Buteurs + minute du but")

print("\n" + "=" * 80)
print("COUVERTURE PAR LE DATASET KAGGLE")
print("=" * 80)
print("  [OUI] Valeur marchande effectif")
print("  [NON] xG match")
print("  [NON] xG cumule saison")
print("  [NON] Lineups par match")
print("  [NON] Buteurs + minutes")

print("\n" + "=" * 80)
print("CALCUL DE LA VALEUR MARCHANDE TOTALE PAR EQUIPE/SAISON")
print("=" * 80)

# Charger quelques exemples pour calculer la valeur totale
examples = [
    "Arsenal_FC_11_2023.csv",
    "Manchester_City_281_2023.csv",
    "Liverpool_FC_31_2023.csv"
]

results = []
for filename in examples:
    csv_files = list(Path(dataset_path).rglob(filename))
    if csv_files:
        df = pd.read_csv(csv_files[0])
        team_name = filename.replace(".csv", "").rsplit("_", 1)[0].replace("_", " ")
        total_value = df['marketValue'].sum()
        player_count = len(df)
        avg_value = df['marketValue'].mean()
        results.append({
            'Equipe': team_name,
            'Saison': 2023,
            'Nb Joueurs': player_count,
            'Valeur Totale (M EUR)': total_value / 1_000_000,
            'Valeur Moyenne (M EUR)': avg_value / 1_000_000
        })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

print("\n" + "=" * 80)
print("DECISION STRATEGIQUE")
print("=" * 80)
print("\nOPTION A : Utiliser uniquement les valeurs marchandes Kaggle")
print("  Avantages :")
print("    - Donnees deja disponibles (1992-2026)")
print("    - Pas besoin de scraping Transfermarkt")
print("    - Facile a integrer")
print("  Inconvenients :")
print("    - Pas de xG (variable cle identifiee)")
print("    - Pas de stats de performance")
print("    - Feature engineering limite")
print("\nOPTION B : Combiner Kaggle + scraping FBref pour xG")
print("  Avantages :")
print("    - Valeurs marchandes : Kaggle (facile)")
print("    - xG : FBref (source fiable)")
print("    - Couverture complete des besoins")
print("  Inconvenients :")
print("    - FBref bloque le scraping (HTTP 403)")
print("    - Necessite une solution alternative (API-Football, autre source)")
print("\nOPTION C : Chercher une autre source de donnees pour xG")
print("  Avantages :")
print("    - Evite les blocages anti-scraping")
print("    - Peut etre plus complet (API structuree)")
print("  Inconvenients :")
print("    - Necessite recherche supplementaire")
print("    - Quotas/limites possibles (API-Football tier gratuit)")

print("\n" + "=" * 80)
print("RECOMMANDATION")
print("=" * 80)
print("\nETAPE 1 : Integrer les valeurs marchandes Kaggle immediatement")
print("  - Creer une fonction d'agregation par equipe/saison")
print("  - Merger avec le dataset existant (pl_features_v1.csv)")
print("  - Feature engineering v2 avec valeurs marchandes")
print("\nETAPE 2 : Explorer API-Football pour les xG")
print("  - Verifier la disponibilite des xG dans l'API")
print("  - Tester le tier gratuit (quotas)")
print("  - Si OK : integrer les xG")
print("  - Si KO : continuer sans xG pour MVP, iteration future")

print("\n" + "=" * 80)
print("Analyse terminee")
print("=" * 80)


BILAN : DATASET KAGGLE vs BESOINS DU PROJET

CONTENU DU DATASET KAGGLE :
--------------------------------------------------------------------------------
Structure :
  - 707 fichiers CSV (equipe + saison)
  - 706 fichiers JSON (identiques aux CSV)
  - 1 fichier clubs.csv (mapping equipes)
  - Periode couverte : 1992-2026

Donnees disponibles par fichier :
  - Liste des joueurs de l'equipe pour la saison
  - Informations joueur : nom, age, poste, nationalite, taille, pied
  - Valeur marchande (marketValue) en EUR
  - Informations transfert : date d'arrivee, club precedent, pret

Donnees ABSENTES :
  - Stats de performance (buts, assists, minutes jouees)
  - xG (Expected Goals)
  - Donnees de matchs
  - Stats par match ou aggregees sur la saison

BESOINS DU PROJET (rappel)

Variables cibles identifiees dans PROJECT_CONTEXT.md :
  1. Valeur marchande effectif (Home/Away) + delta
  2. xG match
  3. xG cumule saison par equipe
  4. Lineups (11 titulaires + bancs)
  5. Buteurs + minute du bu

In [8]:
# Cellule 6 : Consolidation de tous les fichiers joueurs en un seul dataset

import pandas as pd
from pathlib import Path
import re

print("=" * 80)
print("CONSOLIDATION DES FICHIERS JOUEURS KAGGLE")
print("=" * 80)

# Trouver tous les fichiers CSV (sauf clubs.csv)
csv_files = [f for f in Path(dataset_path).rglob("*.csv") if f.name != "clubs.csv"]
print(f"\nNombre de fichiers a traiter : {len(csv_files)}")

# Fonction pour extraire team_name, team_id et season depuis le nom de fichier
def extract_metadata(filename):
    # Format : Team_Name_ID_YEAR.csv
    # Exemple : Arsenal_FC_11_2023.csv
    pattern = r"^(.+)_(\d+)_(\d{4})\.csv$"
    match = re.match(pattern, filename)
    if match:
        team_name = match.group(1).replace("_", " ")
        team_id = int(match.group(2))
        year = int(match.group(3))
        return team_name, team_id, year
    return None, None, None

# Consolider tous les fichiers
all_data = []
errors = []

print("\nTraitement en cours...")
for i, csv_file in enumerate(csv_files, 1):
    if i % 100 == 0:
        print(f"  Traitement : {i}/{len(csv_files)} fichiers")
    
    try:
        team_name, team_id, year = extract_metadata(csv_file.name)
        if team_name is None:
            errors.append(f"Impossible de parser : {csv_file.name}")
            continue
        
        df = pd.read_csv(csv_file)
        df['Team_Name'] = team_name
        df['Team_ID'] = team_id
        df['Season_Year'] = year
        
        # Calculer le code saison (format : "9394", "2324", etc.)
        if year >= 1992 and year < 2000:
            season_code = f"{str(year)[-2:]}{str(year+1)[-2:]}"
        else:
            season_code = f"{str(year)[-2:]}{str(year+1)[-2:]}"
        
        df['Season'] = season_code
        
        all_data.append(df)
    
    except Exception as e:
        errors.append(f"Erreur sur {csv_file.name} : {str(e)}")

# Concatener tous les DataFrames
print(f"\nConcatenation de {len(all_data)} fichiers...")
df_consolidated = pd.concat(all_data, ignore_index=True)

print("\n" + "=" * 80)
print("RESULTATS DE LA CONSOLIDATION")
print("=" * 80)
print(f"Nombre total de joueurs : {len(df_consolidated)}")
print(f"Nombre d'equipes uniques : {df_consolidated['Team_Name'].nunique()}")
print(f"Nombre de saisons uniques : {df_consolidated['Season_Year'].nunique()}")
print(f"Periode couverte : {df_consolidated['Season_Year'].min()} - {df_consolidated['Season_Year'].max()}")
print(f"\nNombre d'erreurs : {len(errors)}")
if errors:
    print("Premieres erreurs :")
    for err in errors[:5]:
        print(f"  - {err}")

print(f"\nColonnes du dataset consolide :")
for col in df_consolidated.columns:
    print(f"  - {col}")

print(f"\nApercu des premieres lignes :")
print(df_consolidated.head())

# Sauvegarder le dataset consolide
output_path = Path("z:/Projets/pl-ldc-prediction/data/external/kaggle_players_all_seasons.csv")
df_consolidated.to_csv(output_path, index=False, encoding='utf-8')

print(f"\n" + "=" * 80)
print(f"Dataset consolide sauvegarde dans :")
print(f"{output_path}")
print(f"Taille : {len(df_consolidated)} lignes x {len(df_consolidated.columns)} colonnes")
print("=" * 80)


CONSOLIDATION DES FICHIERS JOUEURS KAGGLE

Nombre de fichiers a traiter : 706

Traitement en cours...
  Traitement : 100/706 fichiers
  Traitement : 200/706 fichiers
  Traitement : 300/706 fichiers
  Traitement : 400/706 fichiers
  Traitement : 500/706 fichiers
  Traitement : 600/706 fichiers
  Traitement : 700/706 fichiers

Concatenation de 706 fichiers...

RESULTATS DE LA CONSOLIDATION
Nombre total de joueurs : 24541
Nombre d'equipes uniques : 51
Nombre de saisons uniques : 35
Periode couverte : 1992 - 2026

Nombre d'erreurs : 0

Colonnes du dataset consolide :
  - position
  - foot
  - status
  - joinedOn
  - name
  - height
  - id
  - nationality
  - marketValue
  - signedFrom
  - age
  - dateOfBirth
  - currentClub
  - jerseyNumber
  - ageAtSeasonStart
  - isLoan
  - loanedFrom
  - Team_Name
  - Team_ID
  - Season_Year
  - Season

Apercu des premieres lignes :
      position   foot status    joinedOn             name height     id  \
0   Goalkeeper  right    NaN  2013-02-07   Alla

OSError: Cannot save file into a non-existent directory: 'z:/Projets/pl-ldc-prediction/data/external'

In [10]:
# Cellule 6ter : Sauvegarde avec nettoyage des valeurs marchandes

import os

# Nettoyer la colonne marketValue
df_consolidated['marketValue'] = pd.to_numeric(df_consolidated['marketValue'], errors='coerce')

# Determiner le chemin du projet
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
output_dir = project_root / "data" / "external"

# Creer le repertoire si inexistant
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "kaggle_players_all_seasons.csv"
df_consolidated.to_csv(output_path, index=False, encoding='utf-8')

print("=" * 80)
print("SAUVEGARDE DU DATASET CONSOLIDE")
print("=" * 80)
print(f"Fichier sauvegarde : {output_path}")
print(f"Taille : {len(df_consolidated)} lignes x {len(df_consolidated.columns)} colonnes")
print(f"Taille du fichier : {output_path.stat().st_size / (1024*1024):.2f} MB")

print("\n" + "=" * 80)
print("STATISTIQUES GLOBALES")
print("=" * 80)
print(f"Periode : {df_consolidated['Season_Year'].min()} - {df_consolidated['Season_Year'].max()}")
print(f"Nombre d'equipes : {df_consolidated['Team_Name'].nunique()}")
print(f"Nombre de saisons : {df_consolidated['Season'].nunique()}")
print(f"Nombre total de joueurs : {len(df_consolidated)}")

print(f"\nRepartition par saison (top 10 plus recentes) :")
season_counts = df_consolidated.groupby('Season_Year').size().sort_index(ascending=False).head(10)
for year, count in season_counts.items():
    print(f"  {year} : {count} joueurs")

# Statistiques sur marketValue avec gestion des NaN
df_valid_mv = df_consolidated[df_consolidated['marketValue'].notna()]
print(f"\nValeurs marchandes :")
print(f"  Joueurs avec valeur renseignee : {len(df_valid_mv)} / {len(df_consolidated)}")
if len(df_valid_mv) > 0:
    print(f"  Total global : {df_valid_mv['marketValue'].sum() / 1_000_000_000:.2f} Mds EUR")
    print(f"  Moyenne par joueur : {df_valid_mv['marketValue'].mean() / 1_000_000:.2f} M EUR")
    print(f"  Mediane : {df_valid_mv['marketValue'].median() / 1_000_000:.2f} M EUR")
    print(f"  Min : {df_valid_mv['marketValue'].min() / 1_000_000:.2f} M EUR")
    print(f"  Max : {df_valid_mv['marketValue'].max() / 1_000_000:.2f} M EUR")

print("\n" + "=" * 80)
print("Dataset consolide pret")
print("=" * 80)


SAUVEGARDE DU DATASET CONSOLIDE
Fichier sauvegarde : /workspace/data/external/kaggle_players_all_seasons.csv
Taille : 24541 lignes x 21 colonnes
Taille du fichier : 3.77 MB

STATISTIQUES GLOBALES
Periode : 1992 - 2026
Nombre d'equipes : 51
Nombre de saisons : 35
Nombre total de joueurs : 24541

Repartition par saison (top 10 plus recentes) :
  2026 : 595 joueurs
  2025 : 612 joueurs
  2024 : 634 joueurs
  2023 : 853 joueurs
  2022 : 840 joueurs
  2021 : 805 joueurs
  2020 : 801 joueurs
  2019 : 775 joueurs
  2018 : 770 joueurs
  2017 : 738 joueurs

Valeurs marchandes :
  Joueurs avec valeur renseignee : 15193 / 24541
  Total global : 152.95 Mds EUR
  Moyenne par joueur : 10.07 M EUR
  Mediane : 4.00 M EUR
  Min : 0.03 M EUR
  Max : 220.00 M EUR

Dataset consolide pret


In [11]:
# Cellule 7 : Agregation des valeurs marchandes par equipe et saison

import pandas as pd
from pathlib import Path

print("=" * 80)
print("AGREGATION DES VALEURS MARCHANDES PAR EQUIPE/SAISON")
print("=" * 80)

# Agregation par Team_Name et Season
squad_values = df_consolidated.groupby(['Team_Name', 'Season']).agg({
    'marketValue': ['sum', 'mean', 'count'],
    'Season_Year': 'first'
}).reset_index()

# Aplatir les colonnes multi-index
squad_values.columns = ['Team_Name', 'Season', 'Squad_Value_Total', 'Squad_Value_Mean', 'Squad_Size', 'Season_Year']

print(f"\nDimensions du dataset agrege : {squad_values.shape[0]} lignes x {squad_values.shape[1]} colonnes")
print(f"\nApercu des premieres lignes :")
print(squad_values.head(10))

print(f"\n" + "=" * 80)
print("STATISTIQUES SUR LES VALEURS D'EFFECTIF")
print("=" * 80)
print(squad_values[['Squad_Value_Total', 'Squad_Value_Mean', 'Squad_Size']].describe())

print(f"\nTop 10 effectifs les plus chers (toutes saisons confondues) :")
print(squad_values.nlargest(10, 'Squad_Value_Total')[['Team_Name', 'Season', 'Season_Year', 'Squad_Value_Total', 'Squad_Size']])

# Sauvegarder le dataset agrege
output_path = project_root / "data" / "external" / "kaggle_squad_values_by_season.csv"
squad_values.to_csv(output_path, index=False, encoding='utf-8')

print(f"\n" + "=" * 80)
print(f"Dataset agrege sauvegarde dans :")
print(f"{output_path}")
print(f"Taille : {len(squad_values)} lignes x {len(squad_values.columns)} colonnes")
print("=" * 80)


AGREGATION DES VALEURS MARCHANDES PAR EQUIPE/SAISON

Dimensions du dataset agrege : 706 lignes x 6 colonnes

Apercu des premieres lignes :
         Team_Name Season  Squad_Value_Total  Squad_Value_Mean  Squad_Size  \
0  AFC Bournemouth   1516         99900000.0      2.561538e+06          39   
1  AFC Bournemouth   1617        131600000.0      4.112500e+06          32   
2  AFC Bournemouth   1718        144500000.0      4.816667e+06          30   
3  AFC Bournemouth   1819        310000000.0      9.117647e+06          34   
4  AFC Bournemouth   1920        272175000.0      8.505469e+06          32   
5  AFC Bournemouth   2223        287200000.0      7.180000e+06          40   
6  AFC Bournemouth   2324        401750000.0      1.057237e+07          38   
7  AFC Bournemouth   2425        416600000.0      1.542963e+07          27   
8  AFC Bournemouth   2526        659175000.0      1.997500e+07          33   
9  AFC Bournemouth   2627        537875000.0      1.854741e+07          29   

  

In [12]:
# Cellule 8 : Mapping des noms d'equipes et merge avec pl_features_v1.csv

import pandas as pd
from pathlib import Path

print("=" * 80)
print("MAPPING DES NOMS D'EQUIPES KAGGLE <-> PL_FEATURES_V1")
print("=" * 80)

# Charger pl_features_v1.csv
pl_features_path = project_root / "data" / "processed" / "pl_features_v1.csv"
df_pl = pd.read_csv(pl_features_path)

print(f"\nDataset pl_features_v1 charge : {df_pl.shape[0]} lignes x {df_pl.shape[1]} colonnes")
print(f"Periode couverte : {df_pl['Date'].min()} - {df_pl['Date'].max()}")

# Extraire les noms uniques d'equipes dans pl_features_v1
pl_teams = sorted(set(df_pl['HomeTeam'].unique()) | set(df_pl['AwayTeam'].unique()))
print(f"\nNombre d'equipes dans pl_features_v1 : {len(pl_teams)}")

# Extraire les noms uniques d'equipes dans Kaggle
kaggle_teams = sorted(squad_values['Team_Name'].unique())
print(f"Nombre d'equipes dans Kaggle : {len(kaggle_teams)}")

# Creer un mapping automatique (normalisation basique)
def normalize_team_name(name):
    # Supprimer FC, AFC, etc., normaliser espaces
    return name.replace(' FC', '').replace('AFC ', '').replace(' United', '').replace(' City', '').strip()

# Mapping manuel pour les cas problematiques
team_mapping = {
    # Kaggle -> pl_features_v1
    'Arsenal FC': 'Arsenal',
    'Aston Villa': 'Aston Villa',
    'AFC Bournemouth': 'Bournemouth',
    'Brentford FC': 'Brentford',
    'Brighton and Hove Albion': 'Brighton',
    'Burnley FC': 'Burnley',
    'Chelsea FC': 'Chelsea',
    'Crystal Palace': 'Crystal Palace',
    'Everton FC': 'Everton',
    'Fulham FC': 'Fulham',
    'Ipswich Town': 'Ipswich',
    'Leeds United': 'Leeds',
    'Leicester City': 'Leicester',
    'Liverpool FC': 'Liverpool',
    'Luton Town': 'Luton',
    'Manchester City': 'Man City',
    'Manchester United': 'Man United',
    'Newcastle United': 'Newcastle',
    'Norwich City': 'Norwich',
    'Nottingham Forest': "Nott'm Forest",
    'Sheffield United': 'Sheffield United',
    'Southampton FC': 'Southampton',
    'Tottenham Hotspur': 'Tottenham',
    'Watford FC': 'Watford',
    'West Ham United': 'West Ham',
    'Wolverhampton Wanderers': 'Wolves',
    # Equipes historiques
    'Barnsley FC': 'Barnsley',
    'Birmingham City': 'Birmingham',
    'Blackburn Rovers': 'Blackburn',
    'Blackpool FC': 'Blackpool',
    'Bolton Wanderers': 'Bolton',
    'Bradford City': 'Bradford',
    'Cardiff City': 'Cardiff',
    'Charlton Athletic': 'Charlton',
    'Coventry City': 'Coventry',
    'Derby County': 'Derby',
    'Huddersfield Town': 'Huddersfield',
    'Hull City': 'Hull City',
    'Middlesbrough FC': 'Middlesbrough',
    'Oldham Athletic': 'Oldham',
    'Portsmouth FC': 'Portsmouth',
    'Queens Park Rangers': 'QPR',
    'Reading FC': 'Reading',
    'Sheffield Wednesday': 'Sheffield Weds',
    'Stoke City': 'Stoke',
    'Sunderland AFC': 'Sunderland',
    'Swansea City': 'Swansea',
    'Swindon Town': 'Swindon',
    'West Bromwich Albion': 'West Brom',
    'Wigan Athletic': 'Wigan',
    'Wimbledon FC (- 2004)': 'Wimbledon',
}

# Appliquer le mapping
squad_values['Team_Name_Mapped'] = squad_values['Team_Name'].map(team_mapping)

# Verifier les equipes non mappees
unmapped = squad_values[squad_values['Team_Name_Mapped'].isna()]['Team_Name'].unique()
if len(unmapped) > 0:
    print(f"\nATTENTION : {len(unmapped)} equipes Kaggle non mappees :")
    for team in unmapped:
        print(f"  - {team}")
else:
    print(f"\nTous les noms d'equipes Kaggle ont ete mappes avec succes")

print(f"\n" + "=" * 80)
print("VERIFICATION DU MAPPING")
print("=" * 80)
print(f"\nExemple de mapping :")
sample = squad_values[['Team_Name', 'Team_Name_Mapped', 'Season', 'Squad_Value_Total']].head(10)
print(sample)

print("\n" + "=" * 80)
print("Dataset pret pour le merge")
print("=" * 80)


MAPPING DES NOMS D'EQUIPES KAGGLE <-> PL_FEATURES_V1

Dataset pl_features_v1 charge : 12279 lignes x 24 colonnes
Periode couverte : 1993-08-16 - 2026-05-24

Nombre d'equipes dans pl_features_v1 : 51
Nombre d'equipes dans Kaggle : 51

Tous les noms d'equipes Kaggle ont ete mappes avec succes

VERIFICATION DU MAPPING

Exemple de mapping :
         Team_Name Team_Name_Mapped Season  Squad_Value_Total
0  AFC Bournemouth      Bournemouth   1516         99900000.0
1  AFC Bournemouth      Bournemouth   1617        131600000.0
2  AFC Bournemouth      Bournemouth   1718        144500000.0
3  AFC Bournemouth      Bournemouth   1819        310000000.0
4  AFC Bournemouth      Bournemouth   1920        272175000.0
5  AFC Bournemouth      Bournemouth   2223        287200000.0
6  AFC Bournemouth      Bournemouth   2324        401750000.0
7  AFC Bournemouth      Bournemouth   2425        416600000.0
8  AFC Bournemouth      Bournemouth   2526        659175000.0
9  AFC Bournemouth      Bournemouth   262

In [14]:
# Cellule 9bis : Merge avec correction des types

import pandas as pd
from pathlib import Path

print("=" * 80)
print("MERGE DES VALEURS MARCHANDES AVEC PL_FEATURES_V1")
print("=" * 80)

# Convertir Season en string dans les deux datasets pour uniformiser
df_pl['Season'] = df_pl['Season'].astype(str)
squad_values['Season'] = squad_values['Season'].astype(str)

print(f"Types apres conversion :")
print(f"  df_pl['Season'] : {df_pl['Season'].dtype}")
print(f"  squad_values['Season'] : {squad_values['Season'].dtype}")

# Preparer le dataset de valeurs pour le merge
squad_values_for_merge = squad_values[['Team_Name_Mapped', 'Season', 'Squad_Value_Total', 'Squad_Value_Mean', 'Squad_Size']].copy()
squad_values_for_merge.columns = ['Team', 'Season', 'Squad_Value_Total', 'Squad_Value_Mean', 'Squad_Size']

# Merge pour Home Team
df_merged = df_pl.merge(
    squad_values_for_merge,
    left_on=['HomeTeam', 'Season'],
    right_on=['Team', 'Season'],
    how='left',
    suffixes=('', '_drop')
)
df_merged.drop(columns=['Team'], inplace=True)
df_merged.rename(columns={
    'Squad_Value_Total': 'Home_Squad_Value',
    'Squad_Value_Mean': 'Home_Squad_Value_Mean',
    'Squad_Size': 'Home_Squad_Size'
}, inplace=True)

# Merge pour Away Team
df_merged = df_merged.merge(
    squad_values_for_merge,
    left_on=['AwayTeam', 'Season'],
    right_on=['Team', 'Season'],
    how='left',
    suffixes=('', '_drop')
)
df_merged.drop(columns=['Team'], inplace=True)
df_merged.rename(columns={
    'Squad_Value_Total': 'Away_Squad_Value',
    'Squad_Value_Mean': 'Away_Squad_Value_Mean',
    'Squad_Size': 'Away_Squad_Size'
}, inplace=True)

# Creer la feature Delta Squad Value (Home - Away)
df_merged['Squad_Value_Delta'] = df_merged['Home_Squad_Value'] - df_merged['Away_Squad_Value']

print(f"\nDataset merge : {df_merged.shape[0]} lignes x {df_merged.shape[1]} colonnes")
print(f"Nouvelles colonnes ajoutees : 7")
print(f"  - Home_Squad_Value, Home_Squad_Value_Mean, Home_Squad_Size")
print(f"  - Away_Squad_Value, Away_Squad_Value_Mean, Away_Squad_Size")
print(f"  - Squad_Value_Delta")

print(f"\n" + "=" * 80)
print("STATISTIQUES SUR LES NOUVELLES FEATURES")
print("=" * 80)

# Statistiques sur les valeurs manquantes
total_matches = len(df_merged)
home_missing = df_merged['Home_Squad_Value'].isna().sum()
away_missing = df_merged['Away_Squad_Value'].isna().sum()

print(f"\nValeurs manquantes :")
print(f"  Home_Squad_Value : {home_missing} / {total_matches} ({home_missing/total_matches*100:.1f}%)")
print(f"  Away_Squad_Value : {away_missing} / {total_matches} ({away_missing/total_matches*100:.1f}%)")

# Statistiques sur les valeurs renseignees
df_with_values = df_merged[df_merged['Home_Squad_Value'].notna() & df_merged['Away_Squad_Value'].notna()]
print(f"\nMatchs avec valeurs renseignees : {len(df_with_values)} / {total_matches} ({len(df_with_values)/total_matches*100:.1f}%)")

if len(df_with_values) > 0:
    print(f"\nStatistiques sur Squad_Value_Delta (Home - Away) :")
    print(df_with_values['Squad_Value_Delta'].describe())
    
    print(f"\nTop 5 plus grands ecarts de valeur (Home >> Away) :")
    print(df_with_values.nlargest(5, 'Squad_Value_Delta')[['Date', 'HomeTeam', 'AwayTeam', 'FTR', 'Home_Squad_Value', 'Away_Squad_Value', 'Squad_Value_Delta']])
    
    print(f"\nTop 5 plus grands ecarts de valeur (Away >> Home) :")
    print(df_with_values.nsmallest(5, 'Squad_Value_Delta')[['Date', 'HomeTeam', 'AwayTeam', 'FTR', 'Home_Squad_Value', 'Away_Squad_Value', 'Squad_Value_Delta']])

print(f"\n" + "=" * 80)
print("SAUVEGARDE DU DATASET V2")
print("=" * 80)

# Sauvegarder pl_features_v2.csv
output_path = project_root / "data" / "processed" / "pl_features_v2.csv"
df_merged.to_csv(output_path, index=False, encoding='utf-8')

print(f"Fichier sauvegarde : {output_path}")
print(f"Taille : {len(df_merged)} lignes x {len(df_merged.columns)} colonnes")
print(f"Taille du fichier : {output_path.stat().st_size / (1024*1024):.2f} MB")

print("\n" + "=" * 80)
print("INTEGRATION TERMINEE")
print("=" * 80)
print(f"Dataset pl_features_v2.csv cree avec succes")
print(f"Nouvelles features : valeurs marchandes des effectifs Home/Away + Delta")


MERGE DES VALEURS MARCHANDES AVEC PL_FEATURES_V1
Types apres conversion :
  df_pl['Season'] : str
  squad_values['Season'] : str

Dataset merge : 12279 lignes x 31 colonnes
Nouvelles colonnes ajoutees : 7
  - Home_Squad_Value, Home_Squad_Value_Mean, Home_Squad_Size
  - Away_Squad_Value, Away_Squad_Value_Mean, Away_Squad_Size
  - Squad_Value_Delta

STATISTIQUES SUR LES NOUVELLES FEATURES

Valeurs manquantes :
  Home_Squad_Value : 3666 / 12279 (29.9%)
  Away_Squad_Value : 3665 / 12279 (29.8%)

Matchs avec valeurs renseignees : 8558 / 12279 (69.7%)

Statistiques sur Squad_Value_Delta (Home - Away) :
count    8.558000e+03
mean     4.108571e+05
std      2.926578e+08
min     -1.221600e+09
25%     -7.018750e+07
50%      0.000000e+00
75%      6.986250e+07
max      1.221600e+09
Name: Squad_Value_Delta, dtype: float64

Top 5 plus grands ecarts de valeur (Home >> Away) :
            Date  HomeTeam          AwayTeam FTR  Home_Squad_Value  \
8723  2024-04-13  Man City             Luton   H      1.3

In [15]:
# Cellule 10 : Analyse finale et recommandations

import pandas as pd
import matplotlib.pyplot as plt

print("=" * 80)
print("ANALYSE FINALE : INTEGRATION DES VALEURS MARCHANDES KAGGLE")
print("=" * 80)

print("\nRESUME DE L'INTEGRATION")
print("-" * 80)
print(f"Dataset source : kaggle_players_all_seasons.csv")
print(f"  - 24 541 joueurs sur 35 saisons (1992-2026)")
print(f"  - 706 effectifs equipe/saison")
print(f"  - Valeur totale : 152.95 Mds EUR")

print(f"\nDataset cible : pl_features_v1.csv")
print(f"  - 12 279 matchs (2000-2026)")
print(f"  - 24 features de base")

print(f"\nDataset final : pl_features_v2.csv")
print(f"  - 12 279 matchs (2000-2026)")
print(f"  - 31 features (+7 nouvelles)")
print(f"  - Couverture : 69.7% des matchs ont valeurs marchandes renseignees")

print("\n" + "=" * 80)
print("NOUVELLES FEATURES AJOUTEES")
print("=" * 80)
print("1. Home_Squad_Value : Valeur totale effectif domicile (EUR)")
print("2. Home_Squad_Value_Mean : Valeur moyenne par joueur domicile (EUR)")
print("3. Home_Squad_Size : Nombre de joueurs effectif domicile")
print("4. Away_Squad_Value : Valeur totale effectif exterieur (EUR)")
print("5. Away_Squad_Value_Mean : Valeur moyenne par joueur exterieur (EUR)")
print("6. Away_Squad_Size : Nombre de joueurs effectif exterieur")
print("7. Squad_Value_Delta : Difference Home - Away (EUR)")

print("\n" + "=" * 80)
print("COVERAGE TEMPORELLE")
print("=" * 80)

# Analyser la couverture par saison
coverage_by_season = df_merged.groupby('Season').agg({
    'Home_Squad_Value': lambda x: x.notna().sum(),
    'Date': 'count'
}).reset_index()
coverage_by_season['Coverage_Pct'] = (coverage_by_season['Home_Squad_Value'] / coverage_by_season['Date'] * 100).round(1)
coverage_by_season.columns = ['Season', 'With_Values', 'Total_Matches', 'Coverage_Pct']

print("\nCouverture par saison (dernieres 10 saisons) :")
print(coverage_by_season.tail(10).to_string(index=False))

print("\n" + "=" * 80)
print("INSIGHTS PRELIMINAIRES")
print("=" * 80)

# Correlation Squad_Value_Delta avec resultat
df_analysis = df_merged[df_merged['Squad_Value_Delta'].notna()].copy()
df_analysis['Result_Numeric'] = df_analysis['FTR'].map({'H': 1, 'D': 0, 'A': -1})

correlation = df_analysis[['Squad_Value_Delta', 'Result_Numeric']].corr().iloc[0, 1]
print(f"\nCorrelation Squad_Value_Delta vs Resultat : {correlation:.3f}")

# Taux de victoire selon delta positif/negatif/nul
df_analysis['Delta_Sign'] = pd.cut(
    df_analysis['Squad_Value_Delta'],
    bins=[-float('inf'), -1e7, 1e7, float('inf')],
    labels=['Away_Richer', 'Balanced', 'Home_Richer']
)

win_rates = df_analysis.groupby('Delta_Sign')['FTR'].value_counts(normalize=True).unstack(fill_value=0) * 100
print(f"\nTaux de victoire selon equilibre des valeurs :")
print(win_rates.round(1))

print("\n" + "=" * 80)
print("RECOMMANDATIONS POUR MODELISATION V2")
print("=" * 80)
print("\n1. SUBSET OPTIMAL")
print("   - Utiliser les matchs avec valeurs renseignees (8558 matchs, 69.7%)")
print("   - Periode probable : saisons 1516-2627 (2015-2026)")
print("   - Compromis volume/enrichissement favorable")

print("\n2. FEATURES A TESTER EN PRIORITE")
print("   - Squad_Value_Delta : proxy force relative objective")
print("   - Home_Squad_Value / Away_Squad_Value : niveau absolu des equipes")
print("   - Ratio Home/Away : alternative au delta")

print("\n3. FEATURE ENGINEERING AVANCE")
print("   - Squad_Value_Ratio = Home / Away (evite valeurs negatives)")
print("   - Squad_Value_Delta_Norm = Delta / (Home + Away) (normalise par niveau)")
print("   - Interaction Form x Squad_Value : momentum + force")

print("\n4. COMPARAISON V1 vs V2")
print("   - Baseline v1 : 50% accuracy sans valeurs marchandes")
print("   - Objectif v2 : +3-5% accuracy grace aux valeurs")
print("   - Metric cle : amelioration detection Draw (actuellement 8% recall)")

print("\n5. LIMITATIONS IDENTIFIEES")
print("   - Pas de xG (Expected Goals) dans ce dataset Kaggle")
print("   - xG reste la variable prioritaire manquante")
print("   - Solution : exploration API-Football ou autre source xG")

print("\n" + "=" * 80)
print("FICHIERS PRODUITS")
print("=" * 80)
print("1. kaggle_players_all_seasons.csv (3.77 MB)")
print("   - Dataset consolide tous joueurs 1992-2026")
print("2. kaggle_squad_values_by_season.csv")
print("   - Agregation equipe/saison (706 lignes)")
print("3. pl_features_v2.csv (2.08 MB)")
print("   - Dataset enrichi pret pour modelisation v2")

print("\n" + "=" * 80)
print("EXPLORATION KAGGLE TERMINEE")
print("=" * 80)


ANALYSE FINALE : INTEGRATION DES VALEURS MARCHANDES KAGGLE

RESUME DE L'INTEGRATION
--------------------------------------------------------------------------------
Dataset source : kaggle_players_all_seasons.csv
  - 24 541 joueurs sur 35 saisons (1992-2026)
  - 706 effectifs equipe/saison
  - Valeur totale : 152.95 Mds EUR

Dataset cible : pl_features_v1.csv
  - 12 279 matchs (2000-2026)
  - 24 features de base

Dataset final : pl_features_v2.csv
  - 12 279 matchs (2000-2026)
  - 31 features (+7 nouvelles)
  - Couverture : 69.7% des matchs ont valeurs marchandes renseignees

NOUVELLES FEATURES AJOUTEES
1. Home_Squad_Value : Valeur totale effectif domicile (EUR)
2. Home_Squad_Value_Mean : Valeur moyenne par joueur domicile (EUR)
3. Home_Squad_Size : Nombre de joueurs effectif domicile
4. Away_Squad_Value : Valeur totale effectif exterieur (EUR)
5. Away_Squad_Value_Mean : Valeur moyenne par joueur exterieur (EUR)
6. Away_Squad_Size : Nombre de joueurs effectif exterieur
7. Squad_Value_D

# Exploration du fichier epl_raw.csv

In [16]:
# Cellule 11 : Exploration du fichier epl_raw.csv

import pandas as pd
from pathlib import Path

print("=" * 80)
print("EXPLORATION : epl_raw.csv")
print("=" * 80)

# Charger le fichier
epl_path = project_root / "data" / "external" / "epl_raw.csv"

if not epl_path.exists():
    print(f"ERREUR : Fichier introuvable a {epl_path}")
else:
    print(f"Fichier trouve : {epl_path}")
    print(f"Taille : {epl_path.stat().st_size / (1024*1024):.2f} MB")
    
    # Charger avec gestion d'erreurs
    try:
        df_epl = pd.read_csv(epl_path, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df_epl = pd.read_csv(epl_path, encoding='latin1')
        except:
            df_epl = pd.read_csv(epl_path, encoding='iso-8859-1')
    
    print(f"\n" + "=" * 80)
    print("INFORMATIONS GENERALES")
    print("=" * 80)
    print(f"Dimensions : {df_epl.shape[0]} lignes x {df_epl.shape[1]} colonnes")
    
    print(f"\nColonnes disponibles ({len(df_epl.columns)}) :")
    for i, col in enumerate(df_epl.columns, 1):
        dtype = df_epl[col].dtype
        non_null = df_epl[col].notna().sum()
        pct = (non_null / len(df_epl)) * 100
        print(f"  {i:2d}. {col:<30} | Type: {str(dtype):<10} | Non-null: {non_null:>6} ({pct:>5.1f}%)")
    
    print(f"\n" + "=" * 80)
    print("APERCU DES DONNEES")
    print("=" * 80)
    print("\nPremieres lignes :")
    print(df_epl.head(10))
    
    print(f"\nDernieres lignes :")
    print(df_epl.tail(5))
    
    print(f"\n" + "=" * 80)
    print("STATISTIQUES DESCRIPTIVES")
    print("=" * 80)
    print(df_epl.describe(include='all').T)
    
    print(f"\n" + "=" * 80)
    print("RECHERCHE DE VARIABLES CLES")
    print("=" * 80)
    
    # Rechercher des mots-cles pertinents dans les noms de colonnes
    keywords = ['xg', 'goal', 'expected', 'shot', 'pass', 'possession', 'distance', 'minute', 'player', 'lineup']
    
    found = {}
    for keyword in keywords:
        matching = [col for col in df_epl.columns if keyword.lower() in col.lower()]
        if matching:
            found[keyword] = matching
    
    if found:
        print("\nVariables pertinentes trouvees :")
        for keyword, cols in found.items():
            print(f"\n  [{keyword.upper()}]")
            for col in cols:
                print(f"    - {col}")
    else:
        print("\nAucune variable cle identifiee automatiquement")
    
    print(f"\n" + "=" * 80)
    print("Exploration terminee")
    print("=" * 80)


EXPLORATION : epl_raw.csv
Fichier trouve : /workspace/data/external/epl_raw.csv
Taille : 0.52 MB

INFORMATIONS GENERALES
Dimensions : 2280 lignes x 29 colonnes

Colonnes disponibles (29) :
   1. league                         | Type: str        | Non-null:   2280 (100.0%)
   2. season                         | Type: int64      | Non-null:   2280 (100.0%)
   3. game                           | Type: str        | Non-null:   2280 (100.0%)
   4. league_id                      | Type: int64      | Non-null:   2280 (100.0%)
   5. season_id                      | Type: int64      | Non-null:   2280 (100.0%)
   6. game_id                        | Type: int64      | Non-null:   2280 (100.0%)
   7. date                           | Type: str        | Non-null:   2280 (100.0%)
   8. home_team_id                   | Type: int64      | Non-null:   2280 (100.0%)
   9. away_team_id                   | Type: int64      | Non-null:   2280 (100.0%)
  10. home_team                      | Type: str       

In [17]:
# Cellule 12 : Analyse detaillee des variables xG

print("=" * 80)
print("ANALYSE DETAILLEE : epl_raw.csv")
print("=" * 80)

print("\nPERIODE COUVERTE")
print("-" * 80)
print(f"Saisons : {df_epl['season'].min()} - {df_epl['season'].max()}")
print(f"Nombre de saisons : {df_epl['season'].nunique()}")
print(f"Nombre de matchs : {len(df_epl)}")

# Repartition par saison
print(f"\nRepartition par saison :")
season_counts = df_epl.groupby('season').size().sort_index()
for season, count in season_counts.items():
    print(f"  {season} : {count} matchs")

print(f"\n" + "=" * 80)
print("VARIABLES XG DISPONIBLES")
print("=" * 80)

xg_vars = {
    'home_xg': 'Expected Goals domicile (total)',
    'home_np_xg': 'Expected Goals domicile (hors penalties)',
    'home_np_xg_difference': 'Difference buts reels - xG domicile',
    'away_xg': 'Expected Goals exterieur (total)',
    'away_np_xg': 'Expected Goals exterieur (hors penalties)',
    'away_np_xg_difference': 'Difference buts reels - xG exterieur'
}

for var, desc in xg_vars.items():
    mean_val = df_epl[var].mean()
    median_val = df_epl[var].median()
    print(f"\n{var}")
    print(f"  Description : {desc}")
    print(f"  Moyenne : {mean_val:.3f}")
    print(f"  Mediane : {median_val:.3f}")
    print(f"  Min : {df_epl[var].min():.3f}")
    print(f"  Max : {df_epl[var].max():.3f}")

print(f"\n" + "=" * 80)
print("AUTRES VARIABLES INTERESSANTES")
print("=" * 80)

other_vars = {
    'home_expected_points': 'Points attendus domicile',
    'away_expected_points': 'Points attendus exterieur',
    'home_ppda': 'PPDA domicile (pressing intensity)',
    'away_ppda': 'PPDA exterieur (pressing intensity)',
    'home_deep_completions': 'Passes completees zone dangereuse domicile',
    'away_deep_completions': 'Passes completees zone dangereuse exterieur'
}

for var, desc in other_vars.items():
    print(f"\n{var} : {desc}")
    print(f"  Moyenne : {df_epl[var].mean():.3f}")

print(f"\n" + "=" * 80)
print("COMPATIBILITE AVEC PL_FEATURES_V2")
print("=" * 80)

# Verifier les noms d'equipes
epl_teams = sorted(set(df_epl['home_team'].unique()) | set(df_epl['away_team'].unique()))
print(f"\nNombre d'equipes dans epl_raw : {len(epl_teams)}")
print(f"Equipes :")
for team in epl_teams:
    print(f"  - {team}")

print(f"\n" + "=" * 80)
print("FORMAT DES DATES")
print("=" * 80)
print(f"\nExemple de dates :")
print(df_epl['date'].head())
print(f"\nFormat detecte : YYYY-MM-DD HH:MM:SS")

print(f"\n" + "=" * 80)
print("CONCLUSION")
print("=" * 80)
print("\nCe fichier contient :")
print("  1. xG par match (home/away)")
print("  2. xG hors penalties (np_xg)")
print("  3. Expected points par match")
print("  4. Metriques avancees (PPDA, deep completions)")
print("  5. Periode : 2021-2526 (6 saisons)")
print("  6. Couverture : 2280 matchs")
print("\nProchaine etape : Merger avec pl_features_v2.csv")


ANALYSE DETAILLEE : epl_raw.csv

PERIODE COUVERTE
--------------------------------------------------------------------------------
Saisons : 2021 - 2526
Nombre de saisons : 6
Nombre de matchs : 2280

Repartition par saison :
  2021 : 380 matchs
  2122 : 380 matchs
  2223 : 380 matchs
  2324 : 380 matchs
  2425 : 380 matchs
  2526 : 380 matchs

VARIABLES XG DISPONIBLES

home_xg
  Description : Expected Goals domicile (total)
  Moyenne : 1.662
  Mediane : 1.501
  Min : 0.020
  Max : 6.672

home_np_xg
  Description : Expected Goals domicile (hors penalties)
  Moyenne : 1.546
  Mediane : 1.387
  Min : 0.000
  Max : 6.053

home_np_xg_difference
  Description : Difference buts reels - xG domicile
  Moyenne : 0.266
  Mediane : 0.260
  Min : -4.966
  Max : 5.715

away_xg
  Description : Expected Goals exterieur (total)
  Moyenne : 1.368
  Mediane : 1.236
  Min : 0.021
  Max : 5.833

away_np_xg
  Description : Expected Goals exterieur (hors penalties)
  Moyenne : 1.280
  Mediane : 1.135
  Min :

In [26]:
# ========================================================================
# PREPARATION DES DONNEES POUR LE MERGE
# ========================================================================

# 1. Charger pl_features_v2.csv
df_v2 = pd.read_csv('data/processed/pl_features_v2.csv')

# 2. Convertir les dates en format string uniforme YYYY-MM-DD (sans heures)
df_v2['Date'] = pd.to_datetime(df_v2['Date']).dt.strftime('%Y-%m-%d')
df_xg_clean['Date'] = pd.to_datetime(df_xg_clean['Date']).dt.strftime('%Y-%m-%d')

# 3. Harmoniser le type de la colonne Season (convertir en string)
df_v2['Season'] = df_v2['Season'].astype(str)
df_xg_clean['Season'] = df_xg_clean['Season'].astype(str)

# 4. Vérifier les types avant merge
print("Types dans df_v2:")
print(f"  - Date: {df_v2['Date'].dtype}")
print(f"  - Season: {df_v2['Season'].dtype}")
print("\nTypes dans df_xg_clean:")
print(f"  - Date: {df_xg_clean['Date'].dtype}")
print(f"  - Season: {df_xg_clean['Season'].dtype}")

print("\nExemples de valeurs df_v2:")
print(df_v2[['Date', 'Season', 'HomeTeam', 'AwayTeam']].head(3))
print("\nExemples de valeurs df_xg_clean:")
print(df_xg_clean[['Date', 'Season', 'HomeTeam', 'AwayTeam']].head(3))

# ========================================================================
# MERGE AVEC LEFT JOIN
# ========================================================================

df_v3 = df_v2.merge(
    df_xg_clean,
    on=['Date', 'Season', 'HomeTeam', 'AwayTeam'],
    how='left',
    indicator=True
)

# ========================================================================
# ANALYSE DU RESULTAT DU MERGE
# ========================================================================

print("\n" + "="*80)
print("RESULTAT DU MERGE")
print("="*80)

print(f"\nDataset pl_features_v3 cree : {len(df_v3)} lignes x {len(df_v3.columns)} colonnes")

# Analyse du merge indicator
merge_stats = df_v3['_merge'].value_counts()
print("\nStatistiques du merge :")
print(f"  - Matchs sans xG (left_only)  : {merge_stats.get('left_only', 0):,}")
print(f"  - Matchs avec xG (both)       : {merge_stats.get('both', 0):,}")
print(f"  - Taux de couverture xG       : {merge_stats.get('both', 0) / len(df_v3) * 100:.1f}%")

# Supprimer la colonne indicator
df_v3 = df_v3.drop(columns=['_merge'])

# Couverture par saison
coverage_by_season = df_v3.groupby('Season').agg({
    'Home_xG': lambda x: x.notna().sum(),
    'Date': 'count'
}).rename(columns={'Home_xG': 'Matchs_avec_xG', 'Date': 'Total_Matchs'})
coverage_by_season['Couverture_%'] = (coverage_by_season['Matchs_avec_xG'] / coverage_by_season['Total_Matchs'] * 100).round(1)

print("\n" + "-"*80)
print("COUVERTURE xG PAR SAISON")
print("-"*80)
print(coverage_by_season.tail(10))

# ========================================================================
# SAUVEGARDE
# ========================================================================

output_path = 'data/processed/pl_features_v3.csv'
df_v3.to_csv(output_path, index=False)
print(f"\nFichier sauvegarde : {output_path}")
print(f"Taille : {len(df_v3)} lignes x {len(df_v3.columns)} colonnes")

# Liste des nouvelles colonnes
new_cols = [col for col in df_v3.columns if col not in df_v2.columns]
print(f"\nNouvelles colonnes ajoutees ({len(new_cols)}) :")
for col in new_cols:
    print(f"  - {col}")


Types dans df_v2:
  - Date: str
  - Season: str

Types dans df_xg_clean:
  - Date: str
  - Season: str

Exemples de valeurs df_v2:
         Date Season  HomeTeam    AwayTeam
0  2000-08-21      1   Arsenal   Liverpool
1  2000-08-22      1  Bradford     Chelsea
2  2000-08-22      1   Ipswich  Man United

Exemples de valeurs df_xg_clean:
         Date Season        HomeTeam     AwayTeam
0  2020-09-12   2021  Crystal Palace  Southampton
1  2020-09-12   2021          Fulham      Arsenal
2  2020-09-12   2021       Liverpool        Leeds

RESULTAT DU MERGE

Dataset pl_features_v3 cree : 12279 lignes x 42 colonnes

Statistiques du merge :
  - Matchs sans xG (left_only)  : 10,061
  - Matchs avec xG (both)       : 2,218
  - Taux de couverture xG       : 18.1%

--------------------------------------------------------------------------------
COUVERTURE xG PAR SAISON
--------------------------------------------------------------------------------
        Matchs_avec_xG  Total_Matchs  Couverture_%
S

In [24]:
import os
print("Repertoire courant:", os.getcwd())
print("\nContenu du repertoire courant:")
for item in os.listdir('.'):
    print(f"  - {item}")


Repertoire courant: /workspace

Contenu du repertoire courant:
  - .dockerignore
  - .git
  - .gitignore
  - .kiro
  - .pytest_cache
  - app
  - CACHE_MANAGER_IMPLEMENTATION.md
  - config
  - data
  - DATAVALIDATOR_IMPLEMENTATION_SUMMARY.md
  - DEPENDENCIES.md
  - docker-compose.yml
  - Dockerfile
  - FINAL_CHECKPOINT_REPORT.md
  - FINDINGS.md
  - logs
  - models
  - notebooks
  - PROJECT_COMPLETION_SUMMARY.md
  - PROJECT_CONTEXT.md
  - README.md
  - reports
  - requirements-scraping.txt
  - requirements.txt
  - src
  - TASK4_VERIFICATION_REPORT.md
  - TASK_11.1_REQUIREMENTS_REPORT.md
  - TASK_11.2_DOCUMENTATION_REPORT.md
  - TASK_5.1_COMPLETION_REPORT.md
  - TASK_6.1_COMPLETION_REPORT.md
  - TASK_8_CHECKPOINT_REPORT.md
  - TASK_9_CLI_IMPLEMENTATION_REPORT.md
  - tests
  - test_fbref_fallback.py
  - test_fbref_parsing.py
  - test_fbref_urls.py
  - verify_components.py
  - verify_fbref_scraper.py


In [27]:
# ========================================================================
# DIAGNOSTIC : POURQUOI SI PEU DE MATCHS MERGENT ?
# ========================================================================

print("="*80)
print("ANALYSE DES VALEURS DE SEASON")
print("="*80)

# Recharger les données pour diagnostic
df_v2_diag = pd.read_csv('data/processed/pl_features_v2.csv')
df_xg_diag = pd.read_csv('data/external/epl_raw.csv')

print("\nValeurs uniques de Season dans pl_features_v2.csv:")
print(sorted(df_v2_diag['Season'].unique())[:20])  # Premières 20 saisons
print(f"Total: {len(df_v2_diag['Season'].unique())} saisons")

print("\nValeurs uniques de Season dans epl_raw.csv:")
print(sorted(df_xg_diag['season'].unique()))
print(f"Total: {len(df_xg_diag['season'].unique())} saisons")

print("\n" + "-"*80)
print("INTERPRETATION")
print("-"*80)
print("Format Season dans pl_features_v2: code saison (1=2000-2001, 708=2007-2008, etc.)")
print("Format season dans epl_raw: annee de debut (2021=saison 2020-2021)")

print("\n" + "-"*80)
print("CONVERSION NECESSAIRE")
print("-"*80)
print("Il faut convertir 'season' de epl_raw en format code saison pl_features_v2")
print("\nExemples de conversion:")
print("  2020 (epl_raw) -> 2021 -> code '2021' (saison 2020-2021)")
print("  2021 (epl_raw) -> 2122 -> code '2122' (saison 2021-2022)")
print("  2022 (epl_raw) -> 2223 -> code '2223' (saison 2022-2023)")


ANALYSE DES VALEURS DE SEASON

Valeurs uniques de Season dans pl_features_v2.csv:
[np.int64(1), np.int64(102), np.int64(203), np.int64(304), np.int64(405), np.int64(506), np.int64(607), np.int64(708), np.int64(809), np.int64(910), np.int64(1011), np.int64(1112), np.int64(1213), np.int64(1314), np.int64(1415), np.int64(1516), np.int64(1617), np.int64(1718), np.int64(1819), np.int64(1920)]
Total: 33 saisons

Valeurs uniques de Season dans epl_raw.csv:
[np.int64(2021), np.int64(2122), np.int64(2223), np.int64(2324), np.int64(2425), np.int64(2526)]
Total: 6 saisons

--------------------------------------------------------------------------------
INTERPRETATION
--------------------------------------------------------------------------------
Format Season dans pl_features_v2: code saison (1=2000-2001, 708=2007-2008, etc.)
Format season dans epl_raw: annee de debut (2021=saison 2020-2021)

--------------------------------------------------------------------------------
CONVERSION NECESSAIRE
-

In [28]:
# ========================================================================
# ANALYSE DETAILLEE DE LA COUVERTURE xG
# ========================================================================

# Charger le fichier pl_features_v3 qu'on vient de créer
df_v3 = pd.read_csv('data/processed/pl_features_v3.csv')

print("="*80)
print("ANALYSE DE LA COUVERTURE xG DANS pl_features_v3.csv")
print("="*80)

# Statistiques globales
total_matchs = len(df_v3)
matchs_avec_xg = df_v3['Home_xG'].notna().sum()
taux_couverture = (matchs_avec_xg / total_matchs) * 100

print(f"\nStatistiques globales:")
print(f"  - Total matchs         : {total_matchs:,}")
print(f"  - Matchs avec xG       : {matchs_avec_xg:,}")
print(f"  - Matchs sans xG       : {total_matchs - matchs_avec_xg:,}")
print(f"  - Taux de couverture   : {taux_couverture:.1f}%")

# Couverture par saison (toutes les saisons)
print("\n" + "-"*80)
print("COUVERTURE PAR SAISON")
print("-"*80)

coverage = df_v3.groupby('Season').agg({
    'Home_xG': lambda x: x.notna().sum(),
    'Date': 'count'
}).rename(columns={'Home_xG': 'Avec_xG', 'Date': 'Total'})
coverage['Couverture_%'] = (coverage['Avec_xG'] / coverage['Total'] * 100).round(1)
coverage = coverage.sort_index()

# Afficher toutes les saisons avec xG
saisons_avec_xg = coverage[coverage['Avec_xG'] > 0]
print(f"\nSaisons avec donnees xG ({len(saisons_avec_xg)}) :")
print(saisons_avec_xg)

# Vérifier la période couverte
if len(saisons_avec_xg) > 0:
    print(f"\nPeriode xG couverte :")
    print(f"  - Premiere saison : {saisons_avec_xg.index.min()}")
    print(f"  - Derniere saison : {saisons_avec_xg.index.max()}")
    print(f"  - Couverture 100% : {(saisons_avec_xg['Couverture_%'] == 100.0).sum()} saisons")

# Statistiques des valeurs xG
print("\n" + "-"*80)
print("STATISTIQUES DES VALEURS xG")
print("-"*80)

xg_data = df_v3[df_v3['Home_xG'].notna()]
print(f"\nHome_xG (sur {len(xg_data)} matchs) :")
print(f"  - Moyenne : {xg_data['Home_xG'].mean():.3f}")
print(f"  - Mediane : {xg_data['Home_xG'].median():.3f}")
print(f"  - Min     : {xg_data['Home_xG'].min():.3f}")
print(f"  - Max     : {xg_data['Home_xG'].max():.3f}")

print(f"\nAway_xG (sur {len(xg_data)} matchs) :")
print(f"  - Moyenne : {xg_data['Away_xG'].mean():.3f}")
print(f"  - Mediane : {xg_data['Away_xG'].median():.3f}")
print(f"  - Min     : {xg_data['Away_xG'].min():.3f}")
print(f"  - Max     : {xg_data['Away_xG'].max():.3f}")

# Exemple de matchs avec xG
print("\n" + "-"*80)
print("EXEMPLES DE MATCHS AVEC xG")
print("-"*80)
print(xg_data[['Date', 'Season', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'Home_xG', 'Away_xG']].head(10))

print("\n" + "="*80)
print("CONCLUSION")
print("="*80)
print(f"Dataset pl_features_v3.csv cree avec succes")
print(f"  - {len(df_v3.columns)} colonnes (+10 variables xG)")
print(f"  - {matchs_avec_xg:,} matchs avec xG ({taux_couverture:.1f}% du dataset)")
print(f"  - Periode xG : saisons {saisons_avec_xg.index.min() if len(saisons_avec_xg) > 0 else 'N/A'} a {saisons_avec_xg.index.max() if len(saisons_avec_xg) > 0 else 'N/A'}")


ANALYSE DE LA COUVERTURE xG DANS pl_features_v3.csv

Statistiques globales:
  - Total matchs         : 12,279
  - Matchs avec xG       : 2,218
  - Matchs sans xG       : 10,061
  - Taux de couverture   : 18.1%

--------------------------------------------------------------------------------
COUVERTURE PAR SAISON
--------------------------------------------------------------------------------

Saisons avec donnees xG (6) :
        Avec_xG  Total  Couverture_%
Season                              
2021        368    368         100.0
2122        370    370         100.0
2223        370    370         100.0
2324        370    370         100.0
2425        370    370         100.0
2526        370    370         100.0

Periode xG couverte :
  - Premiere saison : 2021
  - Derniere saison : 2526
  - Couverture 100% : 6 saisons

--------------------------------------------------------------------------------
STATISTIQUES DES VALEURS xG
-----------------------------------------------------------

In [29]:
# ========================================================================
# ANALYSE DE CORRELATION : xG vs BUTS REELS
# ========================================================================

import numpy as np
from scipy import stats

# Filtrer les matchs avec xG
df_xg = df_v3[df_v3['Home_xG'].notna()].copy()

print("="*80)
print("CORRELATION xG vs BUTS REELS")
print("="*80)

# 1. Corrélation Home xG vs Home Goals
corr_home = df_xg[['Home_xG', 'FTHG']].corr().iloc[0, 1]
print(f"\nCorrelation Home_xG vs FTHG (buts domicile) : {corr_home:.3f}")

# 2. Corrélation Away xG vs Away Goals
corr_away = df_xg[['Away_xG', 'FTAG']].corr().iloc[0, 1]
print(f"Correlation Away_xG vs FTAG (buts exterieur) : {corr_away:.3f}")

# 3. Différence xG domicile - extérieur vs résultat
df_xg['xG_Delta'] = df_xg['Home_xG'] - df_xg['Away_xG']
df_xg['Goal_Delta'] = df_xg['FTHG'] - df_xg['FTAG']

corr_delta = df_xg[['xG_Delta', 'Goal_Delta']].corr().iloc[0, 1]
print(f"Correlation xG_Delta vs Goal_Delta : {corr_delta:.3f}")

# 4. xG Delta vs résultat du match (H/D/A)
print("\n" + "-"*80)
print("xG DELTA PAR RESULTAT")
print("-"*80)

xg_by_result = df_xg.groupby('FTR')['xG_Delta'].agg(['mean', 'median', 'std', 'count'])
xg_by_result.columns = ['Moyenne', 'Mediane', 'Ecart-type', 'Nombre']
print(xg_by_result)

print("\nInterpretation :")
print("  - Victoire domicile (H) : xG_Delta positif attendu")
print("  - Match nul (D)         : xG_Delta proche de 0 attendu")
print("  - Victoire exterieur (A): xG_Delta negatif attendu")

# 5. Précision des xG : écart moyen
df_xg['Home_xG_Error'] = abs(df_xg['FTHG'] - df_xg['Home_xG'])
df_xg['Away_xG_Error'] = abs(df_xg['FTAG'] - df_xg['Away_xG'])

print("\n" + "-"*80)
print("PRECISION DES xG (erreur absolue moyenne)")
print("-"*80)
print(f"Erreur moyenne Home_xG : {df_xg['Home_xG_Error'].mean():.3f} buts")
print(f"Erreur moyenne Away_xG : {df_xg['Away_xG_Error'].mean():.3f} buts")

# 6. Capacité prédictive : xG_Delta prédit-il le résultat ?
df_xg['xG_Prediction'] = df_xg['xG_Delta'].apply(
    lambda x: 'H' if x > 0.3 else ('A' if x < -0.3 else 'D')
)

accuracy = (df_xg['xG_Prediction'] == df_xg['FTR']).mean()
print("\n" + "-"*80)
print("CAPACITE PREDICTIVE DES xG")
print("-"*80)
print(f"Regle simple : H si xG_Delta > 0.3, A si < -0.3, D sinon")
print(f"Accuracy : {accuracy * 100:.1f}%")

# Matrice de confusion
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(df_xg['FTR'], df_xg['xG_Prediction'], labels=['H', 'D', 'A'])
print("\nMatrice de confusion (lignes=reel, colonnes=predit) :")
print(f"         H     D     A")
print(f"    H  {cm[0,0]:3d}  {cm[0,1]:3d}  {cm[0,2]:3d}")
print(f"    D  {cm[1,0]:3d}  {cm[1,1]:3d}  {cm[1,2]:3d}")
print(f"    A  {cm[2,0]:3d}  {cm[2,1]:3d}  {cm[2,2]:3d}")

print("\n" + "="*80)
print("RESUME")
print("="*80)
print(f"Les xG sont fortement correles aux buts reels (r={corr_home:.2f} dom, r={corr_away:.2f} ext)")
print(f"Le delta xG est correle au delta de buts (r={corr_delta:.2f})")
print(f"Capacite predictive simple : {accuracy*100:.1f}% accuracy")
print(f"\nCes variables seront tres utiles pour le modele de prediction !")


CORRELATION xG vs BUTS REELS

Correlation Home_xG vs FTHG (buts domicile) : 0.623
Correlation Away_xG vs FTAG (buts exterieur) : 0.630
Correlation xG_Delta vs Goal_Delta : 0.682

--------------------------------------------------------------------------------
xG DELTA PAR RESULTAT
--------------------------------------------------------------------------------
      Moyenne   Mediane  Ecart-type  Nombre
FTR                                        
A   -0.769245 -0.716446    1.210856     736
D    0.233007  0.178786    1.038716     527
H    1.172311  1.101491    1.190525     955

Interpretation :
  - Victoire domicile (H) : xG_Delta positif attendu
  - Match nul (D)         : xG_Delta proche de 0 attendu
  - Victoire exterieur (A): xG_Delta negatif attendu

--------------------------------------------------------------------------------
PRECISION DES xG (erreur absolue moyenne)
--------------------------------------------------------------------------------
Erreur moyenne Home_xG : 0.819 

In [30]:
# ========================================================================
# SYNTHESE FINALE : DATASET pl_features_v3.csv
# ========================================================================

print("="*80)
print("SYNTHESE FINALE : ENRICHISSEMENT DU DATASET AVEC xG")
print("="*80)

# Charger les 3 versions pour comparaison
df_v1 = pd.read_csv('data/processed/pl_features_v1.csv')
df_v2 = pd.read_csv('data/processed/pl_features_v2.csv')
df_v3 = pd.read_csv('data/processed/pl_features_v3.csv')

print("\n" + "-"*80)
print("EVOLUTION DES DATASETS")
print("-"*80)

comparison = {
    'Version': ['v1 (baseline)', 'v2 (+valeurs)', 'v3 (+xG)'],
    'Fichier': ['pl_features_v1.csv', 'pl_features_v2.csv', 'pl_features_v3.csv'],
    'Colonnes': [len(df_v1.columns), len(df_v2.columns), len(df_v3.columns)],
    'Lignes': [len(df_v1), len(df_v2), len(df_v3)],
    'Periode': ['2000-2026', '2000-2026', '2000-2026']
}

import pandas as pd
comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

print("\n" + "-"*80)
print("NOUVELLES VARIABLES AJOUTEES")
print("-"*80)

print("\nv1 -> v2 (7 variables valeurs marchandes) :")
v2_new = [col for col in df_v2.columns if col not in df_v1.columns]
for i, col in enumerate(v2_new, 1):
    print(f"  {i}. {col}")

print("\nv2 -> v3 (10 variables xG) :")
v3_new = [col for col in df_v3.columns if col not in df_v2.columns]
for i, col in enumerate(v3_new, 1):
    print(f"  {i}. {col}")

print("\n" + "-"*80)
print("COUVERTURE DES ENRICHISSEMENTS")
print("-"*80)

# Valeurs marchandes
valeurs_coverage = df_v3['Home_Squad_Value'].notna().sum()
valeurs_pct = (valeurs_coverage / len(df_v3)) * 100

# xG
xg_coverage = df_v3['Home_xG'].notna().sum()
xg_pct = (xg_coverage / len(df_v3)) * 100

print(f"\nValeurs marchandes :")
print(f"  - Matchs couverts : {valeurs_coverage:,} / {len(df_v3):,} ({valeurs_pct:.1f}%)")
print(f"  - Periode : saisons 1516-2627 (2015-2026)")

print(f"\nExpected Goals (xG) :")
print(f"  - Matchs couverts : {xg_coverage:,} / {len(df_v3):,} ({xg_pct:.1f}%)")
print(f"  - Periode : saisons 2021-2526 (2020-2026)")

print("\n" + "-"*80)
print("QUALITE DES NOUVELLES VARIABLES")
print("-"*80)

# Corrélations avec résultat (pour les matchs avec données)
df_with_values = df_v3[df_v3['Home_Squad_Value'].notna()].copy()
df_with_xg = df_v3[df_v3['Home_xG'].notna()].copy()

# Convertir FTR en numérique pour corrélation
df_with_values['FTR_num'] = df_with_values['FTR'].map({'H': 1, 'D': 0, 'A': -1})
df_with_xg['FTR_num'] = df_with_xg['FTR'].map({'H': 1, 'D': 0, 'A': -1})

corr_squad_value = df_with_values[['Squad_Value_Delta', 'FTR_num']].corr().iloc[0, 1]
corr_xg = df_with_xg[['Home_xG', 'Away_xG']].copy()
corr_xg['xG_Delta'] = corr_xg['Home_xG'] - corr_xg['Away_xG']
corr_xg['FTR_num'] = df_with_xg['FTR_num']
corr_xg_delta = corr_xg[['xG_Delta', 'FTR_num']].corr().iloc[0, 1]

print(f"\nCorrelation avec resultat du match :")
print(f"  - Squad_Value_Delta : r = {corr_squad_value:.3f}")
print(f"  - xG_Delta          : r = {corr_xg_delta:.3f}")

print("\n" + "="*80)
print("PROCHAINES ETAPES")
print("="*80)
print("\n1. Modelisation avec pl_features_v3.csv")
print("2. Comparer performances modele v1 (baseline) vs v3 (enrichi)")
print("3. Analyser importance des features xG et valeurs marchandes")
print("4. Objectif : passer de 50% a 55-60% accuracy")

print("\n" + "="*80)
print("FICHIERS DISPONIBLES")
print("="*80)
print("\ndata/processed/pl_features_v1.csv : 12,279 matchs x 24 colonnes (baseline)")
print("data/processed/pl_features_v2.csv : 12,279 matchs x 31 colonnes (+valeurs)")
print("data/processed/pl_features_v3.csv : 12,279 matchs x 41 colonnes (+xG)")
print("\ndata/external/kaggle_squad_values_by_season.csv : valeurs par equipe/saison")
print("data/external/epl_raw.csv : donnees xG 2021-2526")


SYNTHESE FINALE : ENRICHISSEMENT DU DATASET AVEC xG

--------------------------------------------------------------------------------
EVOLUTION DES DATASETS
--------------------------------------------------------------------------------
      Version            Fichier  Colonnes  Lignes   Periode
v1 (baseline) pl_features_v1.csv        24   12279 2000-2026
v2 (+valeurs) pl_features_v2.csv        31   12279 2000-2026
     v3 (+xG) pl_features_v3.csv        41   12279 2000-2026

--------------------------------------------------------------------------------
NOUVELLES VARIABLES AJOUTEES
--------------------------------------------------------------------------------

v1 -> v2 (7 variables valeurs marchandes) :
  1. Home_Squad_Value
  2. Home_Squad_Value_Mean
  3. Home_Squad_Size
  4. Away_Squad_Value
  5. Away_Squad_Value_Mean
  6. Away_Squad_Size
  7. Squad_Value_Delta

v2 -> v3 (10 variables xG) :
  1. Home_xG
  2. Home_npxG
  3. Away_xG
  4. Away_npxG
  5. Home_xPoints
  6. Away_xPoi

In [31]:
# Charger le dataset v3 (avec xG)
df_v3 = pd.read_csv('data/processed/pl_features_v3.csv')

print("Dataset pl_features_v3 charge")
print(f"  - {len(df_v3):,} matchs")
print(f"  - {len(df_v3.columns)} colonnes")
print(f"\nMatchs avec xG : {df_v3['Home_xG'].notna().sum():,} ({df_v3['Home_xG'].notna().mean():.1%})")


Dataset pl_features_v3 charge
  - 12,279 matchs
  - 41 colonnes

Matchs avec xG : 2,218 (18.1%)


In [32]:
# Feature 1: xG_Delta (différentiel Home - Away)
df_v3['xG_Delta'] = df_v3['Home_xG'] - df_v3['Away_xG']

# Feature 2: xG_Overperformance (Buts réels - xG)
df_v3['Home_xG_Overperf'] = df_v3['FTHG'] - df_v3['Home_xG']
df_v3['Away_xG_Overperf'] = df_v3['FTAG'] - df_v3['Away_xG']

print("3 nouvelles features creees :")
print("  1. xG_Delta : Home_xG - Away_xG")
print("  2. Home_xG_Overperf : FTHG - Home_xG")
print("  3. Away_xG_Overperf : FTAG - Away_xG")

# Statistiques sur matchs avec xG
df_xg = df_v3[df_v3['Home_xG'].notna()]
print(f"\nStatistiques sur {len(df_xg):,} matchs avec xG :")
print(f"\nxG_Delta :")
print(f"  Mean   : {df_xg['xG_Delta'].mean():+.3f}")
print(f"  Median : {df_xg['xG_Delta'].median():+.3f}")
print(f"  Min/Max: {df_xg['xG_Delta'].min():+.3f} / {df_xg['xG_Delta'].max():+.3f}")
print(f"\nHome_xG_Overperf (efficacite domicile) :")
print(f"  Mean   : {df_xg['Home_xG_Overperf'].mean():+.3f}")
print(f"  Median : {df_xg['Home_xG_Overperf'].median():+.3f}")


3 nouvelles features creees :
  1. xG_Delta : Home_xG - Away_xG
  2. Home_xG_Overperf : FTHG - Home_xG
  3. Away_xG_Overperf : FTAG - Away_xG

Statistiques sur 2,218 matchs avec xG :

xG_Delta :
  Mean   : +0.305
  Median : +0.318
  Min/Max: -5.299 / +5.761

Home_xG_Overperf (efficacite domicile) :
  Mean   : -0.110
  Median : -0.197


In [33]:
# Trier par équipe et date
df_v3 = df_v3.sort_values(['Date']).copy()

# Créer features rolling xG par équipe (5 derniers matchs)
rolling_features = []

for team in df_v3['HomeTeam'].unique():
    # Matchs domicile
    home_mask = df_v3['HomeTeam'] == team
    df_v3.loc[home_mask, 'Home_xG_Form'] = df_v3.loc[home_mask, 'Home_xG'].shift(1).rolling(5, min_periods=1).mean()
    
    # Matchs extérieur
    away_mask = df_v3['AwayTeam'] == team
    df_v3.loc[away_mask, 'Away_xG_Form'] = df_v3.loc[away_mask, 'Away_xG'].shift(1).rolling(5, min_periods=1).mean()

print("Features rolling creees :")
print("  4. Home_xG_Form : Moyenne mobile xG domicile (5 matchs)")
print("  5. Away_xG_Form : Moyenne mobile xG exterieur (5 matchs)")

# Vérifier
df_xg = df_v3[df_v3['Home_xG'].notna()]
print(f"\nStatistiques xG_Form :")
print(f"  Home_xG_Form : {df_xg['Home_xG_Form'].notna().sum():,} valeurs non-NaN")
print(f"  Away_xG_Form : {df_xg['Away_xG_Form'].notna().sum():,} valeurs non-NaN")


Features rolling creees :
  4. Home_xG_Form : Moyenne mobile xG domicile (5 matchs)
  5. Away_xG_Form : Moyenne mobile xG exterieur (5 matchs)

Statistiques xG_Form :
  Home_xG_Form : 2,190 valeurs non-NaN
  Away_xG_Form : 2,190 valeurs non-NaN


In [34]:
# Vérifier les nouvelles colonnes
new_cols = ['xG_Delta', 'Home_xG_Overperf', 'Away_xG_Overperf', 'Home_xG_Form', 'Away_xG_Form']

print("Nouvelles features creees (5) :")
for i, col in enumerate(new_cols, 1):
    non_nan = df_v3[col].notna().sum()
    print(f"  {i}. {col:20s} : {non_nan:,} valeurs ({non_nan/len(df_v3)*100:.1f}%)")

# Sauvegarder pl_features_v4.csv
output_path = 'data/processed/pl_features_v4.csv'
df_v3.to_csv(output_path, index=False)

print(f"\nFichier sauvegarde : {output_path}")
print(f"  - {len(df_v3):,} matchs")
print(f"  - {len(df_v3.columns)} colonnes (+5 vs v3)")
print(f"\nDataset pret pour modelisation v3 dans notebook 06")


Nouvelles features creees (5) :
  1. xG_Delta             : 2,218 valeurs (18.1%)
  2. Home_xG_Overperf     : 2,218 valeurs (18.1%)
  3. Away_xG_Overperf     : 2,218 valeurs (18.1%)
  4. Home_xG_Form         : 2,190 valeurs (17.8%)
  5. Away_xG_Form         : 2,190 valeurs (17.8%)

Fichier sauvegarde : data/processed/pl_features_v4.csv
  - 12,279 matchs
  - 46 colonnes (+5 vs v3)

Dataset pret pour modelisation v3 dans notebook 06
